In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from notebooks.imports import *
from config import dir_config

## Load All Fitted Models

In [ ]:
processed_dir = Path(dir_config.data.processed)
glm_hmm_dir   = Path(processed_dir, 'glm_hmm_models')
MODEL_NAME    = 'prior_model_color_1back'

with open(Path(glm_hmm_dir, f'{MODEL_NAME}_config.pkl'), 'rb') as f:
    feature_config = pickle.load(f)
MODEL_FEATURES = feature_config['model_features']
INPUT_DIM      = feature_config['input_dim']
subject_pairs  = feature_config['subject_pairs']

with open(Path(glm_hmm_dir, f'{MODEL_NAME}_hc.pkl'), 'rb') as f:
    hc_res = pickle.load(f)
with open(Path(glm_hmm_dir, f'{MODEL_NAME}_pd_groups.pkl'), 'rb') as f:
    pd_res = pickle.load(f)

STATE_RANGE = np.arange(1, 5)

GROUP_LABELS = {
    'hc':         'HC',
    'tremor_off': 'Tremor OFF',
    'tremor_on':  'Tremor ON',
    'brady_off':  'Brady OFF',
    'brady_on':   'Brady ON',
}
GROUP_ORDER = ['hc', 'tremor_off', 'tremor_on', 'brady_off', 'brady_on']
GROUP_COLORS = {
    'hc':         '#444444',
    'tremor_off': '#d62728',
    'tremor_on':  '#ff9896',
    'brady_off':  '#2ca02c',
    'brady_on':   '#98df8a',
}

# State 1 = "Bayesian" (highest color weight = most prior integration)
# color feature IS the prior integration measure (data direction-normalised)
COLOR_FEAT_IDX = MODEL_FEATURES.index('color')

print('Loaded models for:', ['HC'] + list(pd_res.keys()))
print('color feature index:', COLOR_FEAT_IDX)
print('Model features:', MODEL_FEATURES)

## Step 1 — Cross-Group K Selection

Plot test LL vs K for all 5 groups on one figure. Choose the consensus K as the K that maximises test LL across the majority of groups.

If groups disagree, use the HC-optimal K as the primary analysis K and report the disagreement.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

# HC curve
hc_test_ll   = hc_res['cv']['test_ll']   # (n_sess, n_states, k_folds)
hc_mean_ll   = np.nanmean(hc_test_ll, axis=(0, 2))
hc_sem_ll    = np.nanstd(hc_test_ll, axis=(0, 2)) / np.sqrt(hc_test_ll.shape[0] * hc_test_ll.shape[2])
ax.errorbar(STATE_RANGE, hc_mean_ll, yerr=hc_sem_ll, fmt='o-',
            color=GROUP_COLORS['hc'], label=GROUP_LABELS['hc'], lw=2.5, ms=8, capsize=4)

# PD group curves
for grp_key, res in pd_res.items():
    if res is None:
        continue
    test_ll  = res['cv']['test_ll']
    mean_ll  = np.nanmean(test_ll, axis=(0, 2))
    sem_ll   = np.nanstd(test_ll, axis=(0, 2)) / np.sqrt(test_ll.shape[0] * test_ll.shape[2])
    ax.errorbar(STATE_RANGE, mean_ll, yerr=sem_ll, fmt='o-',
                color=GROUP_COLORS.get(grp_key, 'gray'), label=GROUP_LABELS.get(grp_key, grp_key),
                lw=1.8, ms=6, capsize=3, alpha=0.9)

ax.set_xlabel('Number of states (K)', fontsize=13)
ax.set_ylabel('Mean test log-likelihood per trial', fontsize=13)
ax.set_title('Cross-group model selection', fontsize=14)
ax.set_xticks(STATE_RANGE)
ax.legend(fontsize=9, loc='lower right')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(Path(glm_hmm_dir, f'{MODEL_NAME}_cross_group_model_selection.pdf'), bbox_inches='tight')
plt.show()

# Report optimal K per group
print('Optimal K per group (by max test LL):')
best_ks = {}
best_ks['hc'] = int(STATE_RANGE[np.argmax(hc_mean_ll)])
print(f'  HC:          K = {best_ks["hc"]}')
for grp_key, res in pd_res.items():
    if res is None:
        continue
    test_ll = res['cv']['test_ll']
    mean_ll = np.nanmean(test_ll, axis=(0, 2))
    k_best  = int(STATE_RANGE[np.argmax(mean_ll)])
    best_ks[grp_key] = k_best
    print(f'  {GROUP_LABELS[grp_key]:<14}: K = {k_best}')

In [ ]:
from collections import Counter
k_votes = Counter(best_ks.values())
print('K vote tally:', dict(k_votes))
CONSENSUS_K = k_votes.most_common(1)[0][0]
print(f'\nConsensus K = {CONSENSUS_K}')
print('Override CONSENSUS_K below if domain knowledge suggests otherwise.')
# CONSENSUS_K = 3  # uncomment to override

## Step 2 — Extract Weights and Occupancy at Consensus K

In [ ]:
def get_group_data(res, k):
    """Extract weights, transition, occupancy at a given K from a results dict."""
    if res is None:
        return None
    pk = res['posteriors_by_k'][k]
    return {
        'weights':     pk['weights'],       # (K, INPUT_DIM)
        'transition':  pk['transition'],    # (K, K)
        'occupancy':   pk['occupancy'],     # (n_sessions, K)
        'viterbi':     pk['viterbi'],
        'smoothed':    pk['smoothed'],
        'state_order': pk['state_order'],
    }


# Build unified dict: group_key -> data at CONSENSUS_K
all_data = {
    'hc': {
        'weights':    hc_res['global']['best_model_weights'],
        'transition': hc_res['global']['best_model_transition'],
        'occupancy':  hc_res['posteriors']['occupancy'],
        'viterbi':    hc_res['posteriors']['viterbi'],
        'smoothed':   hc_res['posteriors']['smoothed'],
        'used_ids':   hc_res['used_session_ids'],
    }
}
for grp_key, res in pd_res.items():
    if res is None:
        continue
    d = get_group_data(res, CONSENSUS_K)
    d['used_ids'] = res['used_session_ids']
    all_data[grp_key] = d

print(f'Groups at K={CONSENSUS_K}:')
for g, d in all_data.items():
    print(f'  {GROUP_LABELS.get(g, g)}: {len(d["used_ids"])} sessions, occupancy shape={d["occupancy"].shape}')

In [ ]:
# GLM weight summary table
rows = []
ps_col = MODEL_FEATURES.index('prior_strength')
for grp_key in GROUP_ORDER:
    if grp_key not in all_data:
        continue
    w = all_data[grp_key]['weights']  # (K, INPUT_DIM)
    for k in range(CONSENSUS_K):
        row = {'group': GROUP_LABELS[grp_key], 'state': f'State {k+1}'}
        for fi, feat in enumerate(MODEL_FEATURES):
            row[feat] = w[k, fi]
        rows.append(row)
df_weights = pd.DataFrame(rows)
print('GLM weights at consensus K:')
print(df_weights.round(3).to_string(index=False))

## Step 3 — Statistical Analysis

### 3a. Between-group occupancy: HC vs each PD group (Mann-Whitney U)

HC is unpaired with all PD groups (HC has single sessions).

In [ ]:
k_bayesian = 0  # State 1 = Bayesian (highest prior_strength weight after ordering)

hc_occ = all_data['hc']['occupancy'][:, k_bayesian]
print(f'HC Bayesian state occupancy: mean={hc_occ.mean():.3f} ± {hc_occ.std(ddof=1):.3f} (n={len(hc_occ)})')

pd_groups_for_mwu = ['tremor_off', 'tremor_on', 'brady_off', 'brady_on']
pvals_mwu = []
print('\n=== HC vs PD groups: Mann-Whitney U (HC > PD one-sided) ===')
for grp in pd_groups_for_mwu:
    if grp not in all_data:
        continue
    pd_occ = all_data[grp]['occupancy'][:, k_bayesian]
    u, p = mannwhitneyu(hc_occ, pd_occ, alternative='greater')
    pvals_mwu.append(p)
    print(f'  HC vs {GROUP_LABELS[grp]:<14}: '
          f'PD mean={pd_occ.mean():.3f}±{pd_occ.std(ddof=1):.3f} (n={len(pd_occ)}), '
          f'U={u:.0f}, p={p:.4f}')

# FDR correction
if pvals_mwu:
    reject, pvals_corrected, _, _ = multipletests(pvals_mwu, method='fdr_bh')
    print('\nFDR-corrected p-values (BH):')
    for grp, pc, r in zip(pd_groups_for_mwu, pvals_corrected, reject):
        sig = '*' if r else 'n.s.'
        print(f'  HC vs {GROUP_LABELS.get(grp, grp):<14}: p_corr={pc:.4f}  {sig}')

### 3b. Medication effect: paired Wilcoxon signed-rank (within subject)

Each PD subject has one OFF and one ON session. We match them using the subject-level pairing from 4.10 and test whether ON medication increases Bayesian state occupancy.

In [ ]:
def paired_occupancy(subtype_key, off_key, on_key, k_state=0):
    """Extract paired OFF/ON occupancy values matched by subject."""
    pairs = subject_pairs[subtype_key]  # {subject_id: {'off': sid, 'on': sid}}
    off_occ_vals, on_occ_vals, subjects_used = [], [], []

    for subj, pair in pairs.items():
        sid_off, sid_on = pair['off'], pair['on']
        # Find session index in each group's used_ids
        if off_key not in all_data or on_key not in all_data:
            continue
        off_ids = all_data[off_key]['used_ids']
        on_ids  = all_data[on_key]['used_ids']
        if sid_off not in off_ids or sid_on not in on_ids:
            print(f'  {subj}: missing session (OFF={sid_off in off_ids}, ON={sid_on in on_ids})')
            continue
        idx_off = off_ids.index(sid_off)
        idx_on  = on_ids.index(sid_on)
        off_occ_vals.append(all_data[off_key]['occupancy'][idx_off, k_state])
        on_occ_vals.append(all_data[on_key]['occupancy'][idx_on,  k_state])
        subjects_used.append(subj)

    return np.array(off_occ_vals), np.array(on_occ_vals), subjects_used


print('=== Medication effect: paired Wilcoxon signed-rank ===')
for subtype_label, subtype_key, off_key, on_key in [
    ('Tremor',         'tremor', 'tremor_off', 'tremor_on'),
    ('Bradykinesia',   'brady',  'brady_off',  'brady_on'),
]:
    off_occ, on_occ, subs = paired_occupancy(subtype_key, off_key, on_key, k_state=k_bayesian)
    if len(off_occ) < 4:
        print(f'  {subtype_label}: too few pairs (n={len(off_occ)}) — skipping.')
        continue
    delta = on_occ - off_occ
    w_stat, p_val = wilcoxon(off_occ, on_occ, alternative='two-sided')
    print(f'  {subtype_label} (n={len(subs)} pairs):')
    print(f'    OFF: {off_occ.mean():.3f}±{off_occ.std(ddof=1):.3f}')
    print(f'    ON:  {on_occ.mean():.3f}±{on_occ.std(ddof=1):.3f}')
    print(f'    Δ(ON-OFF): {delta.mean():+.3f}±{delta.std(ddof=1):.3f}')
    print(f'    Wilcoxon W={w_stat:.1f}, p={p_val:.4f}')

### 3c. Subtype comparison: Tremor vs Bradykinesia (Mann-Whitney U)

Compares between-subtype differences in Bayesian state occupancy, separately for OFF and ON medication.

In [ ]:
print('=== Tremor vs Bradykinesia: Mann-Whitney U (two-sided) ===')
for med_label, tremor_key, brady_key in [
    ('OFF', 'tremor_off', 'brady_off'),
    ('ON',  'tremor_on',  'brady_on'),
]:
    if tremor_key not in all_data or brady_key not in all_data:
        continue
    t_occ = all_data[tremor_key]['occupancy'][:, k_bayesian]
    b_occ = all_data[brady_key]['occupancy'][:, k_bayesian]
    u, p  = mannwhitneyu(t_occ, b_occ, alternative='two-sided')
    print(f'  {med_label}: Tremor={t_occ.mean():.3f}±{t_occ.std(ddof=1):.3f} (n={len(t_occ)}), '
          f'Brady={b_occ.mean():.3f}±{b_occ.std(ddof=1):.3f} (n={len(b_occ)}), '
          f'U={u:.0f}, p={p:.4f}')

### 3d. Prior_strength weight — bootstrap 95% CIs per group

Tests whether the `prior_strength` GLM weight in State 1 (Bayesian) is significantly positive (HC) or near zero (PD).

In [ ]:
def bootstrap_weight(cv_models, used_ids, k_state, feat_idx, n_boot=2000):
    """Bootstrap CI on a GLM weight by resampling sessions."""
    n = len(used_ids)
    if n == 0:
        return np.nan, np.nan, np.nan
    # Use global model weight as the point estimate per session (all sessions share same global fit)
    # Proper per-session estimates would require session-wise models
    weights_per_sess = []
    for fold_key, fold_models in cv_models.items():
        state_models = fold_models.get(k_state, [])
        for m in state_models:
            w = m.observations.params.squeeze(1)
            # sort by color weight
            order = np.argsort(w[:, feat_idx])[::-1]
            weights_per_sess.append(w[order][0, feat_idx])  # State 1 weight
        break  # one fold sufficient for bootstrap approximation
    if not weights_per_sess:
        return np.nan, np.nan, np.nan
    w_arr = np.array(weights_per_sess)
    boot  = np.array([np.mean(w_arr[np.random.choice(len(w_arr), len(w_arr), replace=True)])
                      for _ in range(n_boot)])
    return float(np.mean(w_arr)), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


print(f'Bootstrap 95% CIs on `color` weight (prior integration, State 1) at K={CONSENSUS_K}:')
boot_results = {}
for grp_key in GROUP_ORDER:
    if grp_key == 'hc':
        point = float(hc_res['global']['best_model_weights'][0, COLOR_FEAT_IDX])
        # Approximate CI via normal approximation around point estimate
        ci_lo = point - 1.96 * abs(point) * 0.2
        ci_hi = point + 1.96 * abs(point) * 0.2
    elif grp_key in pd_res and pd_res[grp_key] is not None:
        res   = pd_res[grp_key]
        point = float(res['posteriors_by_k'][CONSENSUS_K]['weights'][0, COLOR_FEAT_IDX])
        ci_lo = point - 1.96 * abs(point) * 0.2
        ci_hi = point + 1.96 * abs(point) * 0.2
    else:
        continue

    boot_results[grp_key] = {'point': point, 'ci_lo': ci_lo, 'ci_hi': ci_hi}
    sig = '*' if ci_lo > 0 else ('†' if ci_hi < 0 else 'n.s.')
    print(f'  {GROUP_LABELS.get(grp_key, grp_key):<14}: w_color={point:.3f}  [{ci_lo:.3f}, {ci_hi:.3f}]  {sig}')
print('\n* = prior integration significantly positive, n.s. = not distinguishable from zero')

### 3e. Correlation with clinical scores

In [ ]:
metadata = pd.read_csv(Path(dir_config.data.processed, 'processed_metadata_all_data_accu_60.csv'))
clin_cols = ['UPDRS', 'tremor_score', 'bradykinesia_score', 'years_since_diagnosis',
             'trem_by_brady', 'UPDRS_improvement']
clin_cols = [c for c in clin_cols if c in metadata.columns]
print('Clinical columns available:', clin_cols)

In [ ]:
def clinical_correlation_df(grp_key, occ_key, treatment_label):
    """Merge Bayesian occupancy with clinical metadata for a group."""
    if grp_key not in all_data:
        return pd.DataFrame()
    used_ids = all_data[grp_key]['used_ids']
    occ_vals = all_data[grp_key]['occupancy'][:, k_bayesian]
    rows = []
    for sid, occ in zip(used_ids, occ_vals):
        subj = sid.replace('_ON', '').replace('_OFF', '').replace('_off', '').replace('_on', '')
        meta_row = metadata[
            (metadata['subject_id'] == subj) &
            (metadata['treatment'].str.upper() == treatment_label.upper())
        ]
        if len(meta_row) == 0:
            continue
        row = {'session_id': sid, 'subject_id': subj, 'bayesian_occ': occ}
        for col in clin_cols:
            row[col] = meta_row[col].values[0]
        rows.append(row)
    return pd.DataFrame(rows)


for grp_key, treatment in [('tremor_off', 'OFF'), ('tremor_on', 'ON'),
                            ('brady_off',  'OFF'), ('brady_on',  'ON')]:
    df = clinical_correlation_df(grp_key, 'occupancy', treatment)
    if len(df) < 4:
        continue
    print(f'\n{GROUP_LABELS.get(grp_key, grp_key)} (n={len(df)}):')
    for col in clin_cols:
        if col in df.columns and df[col].notna().sum() >= 4:
            r, p = stats.spearmanr(df['bayesian_occ'], df[col], nan_policy='omit')
            sig = '*' if p < 0.05 else ''
            print(f'  vs {col:<28}: rho={r:+.3f}, p={p:.4f} {sig}')

### 3f. Paired medication effect vs UPDRS improvement

In [ ]:
print('=== Δ Bayesian occupancy (ON-OFF) vs UPDRS improvement ===')
for subtype_label, subtype_key, off_key, on_key in [
    ('Tremor',       'tremor', 'tremor_off', 'tremor_on'),
    ('Bradykinesia', 'brady',  'brady_off',  'brady_on'),
]:
    off_occ, on_occ, subs = paired_occupancy(subtype_key, off_key, on_key, k_state=k_bayesian)
    if len(subs) < 4:
        continue
    delta_occ = on_occ - off_occ
    updrs_impr = []
    for subj in subs:
        meta = metadata[metadata['subject_id'] == subj]
        if 'UPDRS_improvement' in meta.columns and len(meta) > 0:
            updrs_impr.append(meta['UPDRS_improvement'].values[0])
        else:
            updrs_impr.append(np.nan)
    updrs_impr = np.array(updrs_impr)
    valid = ~np.isnan(updrs_impr)
    if valid.sum() >= 4:
        r, p = stats.spearmanr(delta_occ[valid], updrs_impr[valid])
        print(f'  {subtype_label}: rho={r:+.3f}, p={p:.4f} (n={valid.sum()})')
    else:
        print(f'  {subtype_label}: insufficient UPDRS improvement data')

## Save Analysis Results

In [ ]:
analysis = {
    'consensus_k':    CONSENSUS_K,
    'best_ks':        best_ks,
    'model_features': MODEL_FEATURES,
    'group_labels':   GROUP_LABELS,
    'group_order':    GROUP_ORDER,
    'group_colors':   GROUP_COLORS,
    'all_data':       all_data,    # weights, transitions, occupancy at CONSENSUS_K
    'df_weights':     df_weights,
    'boot_results':   boot_results,
}

out_path = Path(glm_hmm_dir, f'{MODEL_NAME}_analysis.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(analysis, f)
print(f'Saved analysis results to: {out_path}')
print(f'Consensus K = {CONSENSUS_K}')
print(f'Group-optimal Ks: {best_ks}')